# Convergence re-run of the glioma graph benchmark

Lifts the 30-epoch cap and restores reference capacity. Everything else - split, seeds,
batch size, learning rate, loss - is identical to the published run.

**Before running:** enable the GPU (Kaggle: *Settings -> Accelerator -> GPU*; Colab:
*Runtime -> Change runtime type -> T4 GPU*), and make the three parquet files reachable.
Read `README_gpu_convergence.md` for what the outcomes mean.


## 1. Dependencies

In [ ]:
!pip -q install torch-geometric rdkit-pypi
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Bundle and data

Upload `gpu_convergence.tar.gz` and the three parquet files. On Kaggle put them in a Dataset and
point `DATA` at `/kaggle/input/<your-dataset>`; on Colab upload to the session or mount Drive.

In [ ]:
import os, tarfile, shutil
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)

BUNDLE = "gpu_convergence.tar.gz"      # adjust if you placed it elsewhere
DATA   = "."                           # directory holding the three .parquet files

if not os.path.isdir("gpu_convergence"):
    tarfile.open(BUNDLE).extractall(".")
os.chdir("gpu_convergence")

os.makedirs("data", exist_ok=True)
for f in ["chembl_multitask_matrix.parquet", "decoy_set.parquet", "cmnpd_curated.parquet"]:
    src = os.path.join(DATA, f)
    assert os.path.exists(src), "missing " + src
    shutil.copy(src, "data/" + f)
print(sorted(os.listdir("data")))

## 3. Verify the GPU and measure per-epoch cost

Do not skip this - it tells you whether the full run fits your session.

In [ ]:
!python smoke_test.py

## 4. Build the graph cache (once, ~10-20 min)

On Kaggle, save the finished `cache/` directory as a Dataset output so later sessions skip this step.

In [ ]:
!python src/build_cache.py
!python src/build_flat.py

## 5. Train

15 runs (5 architectures x 3 seeds). Completed runs are skipped, so re-running this cell after a
session timeout resumes where it stopped.

In [ ]:
!bash run_convergence.sh

## 6. Archive before the session ends

`metrics/` holds the per-epoch validation histories. The published run lost these; do not repeat that.

In [ ]:
!tar czf converged_results.tar.gz metrics models runs_converged
import os; print("%.1f MB" % (os.path.getsize("converged_results.tar.gz")/1e6))

In [ ]:
# Quick look: where did early stopping actually fire?
import glob, json
rows = []
for p in sorted(glob.glob("metrics/*_run.json")):
    r = json.load(open(p))
    rows.append((r["arch"], r["seed"], r["epochs_run"], r["best_val_loss"],
                 r["params"], r["wall_s"] / 60))
print("%-12s %5s %7s %10s %10s %8s" % ("arch", "seed", "epochs", "best_val", "params", "min"))
for r in rows:
    print("%-12s %5d %7d %10.4f %10d %8.1f" % r)
print()
print("runs that hit the 300-epoch ceiling (still not converged):",
      sum(1 for r in rows if r[2] >= 300))